# Co-occurrence & Graphe de Synergies — v2 (pondérée)

Améliorations par rapport à v1 :
- **Pondération par placement** : un deck 1er place pèse plus qu'un deck 50e (`1 / placement`)
- **Pondération temporelle** : les decks récents pèsent plus (décroissance exponentielle, demi-vie 365 jours)
- **Quantités réelles** : jouer 3x une carte ≠ 1x (amount normalisé sur 3)
- **Side deck séparé** : co-occurrence side deck → signal sur les menaces anticipées par les joueurs

La table `card_cooccurrence` est reconstruite à partir de ces données qualifiées.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

con = sqlite3.connect('../data/yugioh.db')

# Charger toutes les zones avec placement et date
df = pd.read_sql("""
    SELECT dc.deck_id, dc.card_name, dc.amount, dc.zone,
           td.archetype, td.placement, td.uploaded
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0
""", con)

print(f'Lignes chargées   : {len(df):,}')
print(f'Decks uniques     : {df["deck_id"].nunique():,}')
print(f'Cartes uniques    : {df["card_name"].nunique():,}')
print(f'Zones             : {df["zone"].value_counts().to_dict()}')

## 1. Poids par deck (placement × recency)

In [ ]:
REFERENCE_DATE = datetime(2026, 6, 14)
DECAY_DAYS = 365        # demi-vie : un deck vieux d'1 an pèse ~37% d'un deck récent
MAX_PLACEMENT_WEIGHT = 1.0

def placement_weight(p):
    """1er place → 1.0, 2e → 0.5, 10e → 0.1. NULL → 0.3 (poids neutre bas)."""
    if pd.isna(p) or p <= 0:
        return 0.3
    return min(MAX_PLACEMENT_WEIGHT, 1.0 / p)

def recency_weight(uploaded_str):
    """Décroissance exponentielle : exp(-jours / DECAY_DAYS)."""
    if not uploaded_str:
        return 0.5
    try:
        d = datetime.fromisoformat(str(uploaded_str)[:10])
        days_old = max(0, (REFERENCE_DATE - d).days)
        return float(np.exp(-days_old / DECAY_DAYS))
    except Exception:
        return 0.5

# Calculer les poids par deck (une ligne par deck)
deck_meta = (df[['deck_id', 'placement', 'uploaded']]
             .drop_duplicates('deck_id')
             .set_index('deck_id'))

deck_meta['w_placement'] = deck_meta['placement'].apply(placement_weight)
deck_meta['w_recency']   = deck_meta['uploaded'].apply(recency_weight)
deck_meta['weight']      = deck_meta['w_placement'] * deck_meta['w_recency']

# Normaliser : poids moyen = 1 (les scores Jaccard restent comparables à v1)
deck_meta['weight'] = deck_meta['weight'] / deck_meta['weight'].mean()

print('Distribution des poids de decks :')
print(deck_meta['weight'].describe().round(3))
print()
print(f'Deck poids min : {deck_meta["weight"].min():.3f}')
print(f'Deck poids max : {deck_meta["weight"].max():.3f}')

## 2. Matrice pondérée — main deck

In [ ]:
# ── Main deck uniquement ──────────────────────────────────────────────────────
main = df[df['zone'] == 'main'].copy()
main = main.merge(deck_meta[['weight']], left_on='deck_id', right_index=True)

# Pondération par quantité : amount normalisé sur 3 (max copies légal)
# card_score(deck, card) = (amount / 3) * deck_weight
main['score'] = (main['amount'].clip(upper=3) / 3.0) * main['weight']

# Matrice : decks × cartes  (score pondéré)
W = main.groupby(['deck_id', 'card_name'])['score'].max().unstack(fill_value=0.0)

# Filtrer : garder seulement les cartes présentes dans au moins 10 decks
card_presence = (W > 0).sum()
W = W[card_presence[card_presence >= 10].index]

print(f'Matrice pondérée main deck : {W.shape[0]} decks × {W.shape[1]} cartes')

## 3. Co-occurrence pondérée + Jaccard

In [ ]:
# co-occurrence pondérée : W.T @ W
# (W.T @ W)[i,j] = Σ_d  score(d,i) * score(d,j)
m = W.values
cooc_weighted = m.T @ m                      # (nb_cartes × nb_cartes)

# Pour le Jaccard pondéré :
#   count_weighted[i] = Σ_d score(d,i)^2   ← diagonale de cooc_weighted
#   jaccard[i,j] = cooc[i,j] / (count[i] + count[j] - cooc[i,j])
card_counts = np.diag(cooc_weighted)
union = card_counts[:, None] + card_counts[None, :] - cooc_weighted
jaccard = np.where(union > 0, cooc_weighted / union, 0.0)
np.fill_diagonal(jaccard, 0.0)

# Co-occurrence brute (nombre de decks où les deux cartes sont présentes ensemble)
binary = (W.values > 0).astype(np.float32)
cooc_count = (binary.T @ binary).astype(int)

cards = W.columns.tolist()
print(f'Calcul terminé. Shape jaccard : {jaccard.shape}')
print(f'Jaccard max (hors diag) : {np.max(jaccard):.3f}')
print(f'Jaccard moyen (> 0)     : {jaccard[jaccard > 0].mean():.3f}')

## 4. Top paires

In [ ]:
upper = np.triu(jaccard, k=1)
pairs_idx = np.argwhere(upper > 0.1)

pairs = []
for i, j in pairs_idx:
    pairs.append({
        'card_a':     cards[i],
        'card_b':     cards[j],
        'jaccard':    round(float(jaccard[i, j]), 4),
        'cooc_count': int(cooc_count[i, j])
    })

pairs_df = pd.DataFrame(pairs).sort_values('jaccard', ascending=False)
print(f'Paires avec Jaccard > 0.1 : {len(pairs_df):,}')
print()
print('Top 20 paires :')
pairs_df.head(20)

## 5. Co-occurrence par archetype

In [ ]:
def top_pairs_for_archetype(archetype, min_jaccard=0.5, top_n=15):
    """Co-occurrence pondérée sur les decks d'un archetype donné."""
    deck_ids = df[(df['zone'] == 'main') & (df['archetype'] == archetype)]['deck_id'].unique()
    sub_W = W.loc[W.index.isin(deck_ids)]
    sub_W = sub_W.loc[:, (sub_W > 0).sum() > 0]

    if sub_W.shape[0] < 5:
        print(f'Pas assez de decks pour {archetype} ({sub_W.shape[0]})')
        return

    m2 = sub_W.values
    c2 = m2.T @ m2
    d2 = np.diag(c2)
    u2 = d2[:, None] + d2[None, :] - c2
    j2 = np.where(u2 > 0, c2 / u2, 0.0)
    np.fill_diagonal(j2, 0.0)

    bin2 = (sub_W.values > 0).astype(np.float32)
    cnt2 = (bin2.T @ bin2).astype(int)

    local_cards = sub_W.columns.tolist()
    idx2 = np.argwhere(np.triu(j2, k=1) >= min_jaccard)
    result = [{'card_a': local_cards[i], 'card_b': local_cards[j],
               'jaccard': round(float(j2[i,j]), 3), 'count': int(cnt2[i,j])}
              for i, j in idx2]

    return pd.DataFrame(result).sort_values('jaccard', ascending=False).head(top_n)

print('=== Maliss ===')
display(top_pairs_for_archetype('Maliss'))
print('=== Tenpai Dragon ===')
display(top_pairs_for_archetype('Tenpai Dragon'))
print('=== Snake-Eye ===')
display(top_pairs_for_archetype('Snake-Eye'))

## 6. Staples universelles

In [ ]:
# Fréquence pondérée par deck : sum des scores / sum des poids de tous les decks
total_weight = deck_meta.loc[W.index, 'weight'].sum()
card_freq_weighted = W.sum(axis=0) / total_weight

staples = card_freq_weighted[card_freq_weighted > 0.2].sort_values(ascending=False)
print(f'Cartes présentes dans >20% des decks (pondéré) : {len(staples)}')
for card, freq in staples.items():
    print(f'  {freq:.0%}  {card}')

## 7. Side deck — menaces anticipées

In [ ]:
# Le side deck révèle quelles cartes les joueurs mettent pour contrer la méta
side = df[df['zone'] == 'side'].copy()
side = side.merge(deck_meta[['weight']], left_on='deck_id', right_index=True)
side['score'] = (side['amount'].clip(upper=3) / 3.0) * side['weight']

W_side = side.groupby(['deck_id', 'card_name'])['score'].max().unstack(fill_value=0.0)

# Filtrer : au moins 5 decks
card_presence_side = (W_side > 0).sum()
W_side = W_side[card_presence_side[card_presence_side >= 5].index]

print(f'Matrice side deck : {W_side.shape[0]} decks × {W_side.shape[1]} cartes')
print()

# Co-occurrence side
ms = W_side.values
cooc_side = ms.T @ ms
cs = np.diag(cooc_side)
us = cs[:, None] + cs[None, :] - cooc_side
jaccard_side = np.where(us > 0, cooc_side / us, 0.0)
np.fill_diagonal(jaccard_side, 0.0)

bin_s = (W_side.values > 0).astype(np.float32)
count_side = (bin_s.T @ bin_s).astype(int)

cards_side = W_side.columns.tolist()

# Cartes les plus présentes en side
total_weight_side = deck_meta.loc[deck_meta.index.isin(W_side.index), 'weight'].sum()
side_freq = W_side.sum(axis=0) / total_weight_side
top_side = side_freq.sort_values(ascending=False).head(30)

print('Top 30 cartes en side deck (pondéré) :')
for card, freq in top_side.items():
    print(f'  {freq:.0%}  {card}')

## 8. Sauvegarder en base

In [ ]:
con2 = sqlite3.connect('../data/yugioh.db')

# ── Main deck co-occurrence ────────────────────────────────────────────────────
significant = pairs_df[pairs_df['jaccard'] > 0.05].copy()

con2.execute("DROP TABLE IF EXISTS card_cooccurrence")
con2.execute("""
    CREATE TABLE card_cooccurrence (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
significant.to_sql('card_cooccurrence', con2, if_exists='append', index=False)

# ── Side deck co-occurrence ────────────────────────────────────────────────────
upper_s = np.triu(jaccard_side, k=1)
pairs_side_idx = np.argwhere(upper_s > 0.05)
pairs_side = [{'card_a': cards_side[i], 'card_b': cards_side[j],
               'jaccard': round(float(jaccard_side[i,j]), 4),
               'cooc_count': int(count_side[i,j])}
              for i, j in pairs_side_idx]
pairs_side_df = pd.DataFrame(pairs_side)

con2.execute("DROP TABLE IF EXISTS card_cooccurrence_side")
con2.execute("""
    CREATE TABLE card_cooccurrence_side (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
if not pairs_side_df.empty:
    pairs_side_df.to_sql('card_cooccurrence_side', con2, if_exists='append', index=False)

con2.commit()
con2.close()

print(f'✓ {len(significant):,} paires main deck sauvegardées dans card_cooccurrence')
print(f'✓ {len(pairs_side_df):,} paires side deck sauvegardées dans card_cooccurrence_side')